# 📦 Resource Management

Create, read, update, and relate resources in the Interfacer ecosystem.
Based on the **ValueFlows** economic vocabulary.

## ValueFlows Concepts

| Concept | Description |
|---|---|
| **EconomicResource** | A thing — project, machine, material, DPP |
| **EconomicEvent** | An action — creation, transfer, consumption |
| **Process** | A workflow connecting events — "creation of X" |
| **ResourceSpecification** | A type/template — "Design", "Machine" |
| **Proposal** | A contribution request — fork → modify → merge |

```mermaid
graph LR
    P[Process] -->|produces| R[EconomicResource]
    P -->|consumes| R2[Existing Resource]
    R -->|classifiedAs| T[ResourceSpecification]
    R -->|locatedAt| L[Location]
```

## Setup: Import & Auth

> ⚠️ You must be authenticated before creating resources.
> If you haven't run `01_Authentication_and_Crypto.ipynb` yet,
> run the auth flow cells below first.

In [ ]:
(async () => {
  const { InterfacerClient, ProjectType } = await import('https://esm.sh/@dyne/interfacer-client');
  const { SANDBOX_CONFIG } = await import('/files/config.js');
  const client = new InterfacerClient(SANDBOX_CONFIG);
  console.log('🔐 Authenticated:', client.isAuthenticated());
  console.log('👤 User ID:', client.store.getItem('authId') || '(not authenticated)');
  if (!client.isAuthenticated()) {
    console.log('\n⚠️ Not authenticated! Run 01_Authentication_and_Crypto.ipynb first.');
    console.log('Or authenticate here:');
    console.log('  const seed = client.store.getItem("seed");');
    console.log('  const email = client.store.getItem("authEmail");');
    console.log('  if (seed && email) {');
    console.log('    const hmac = await client.auth.requestHmac(email, false);');
    console.log('    await client.auth.recreateKeys(seed, hmac);');
    console.log('    await client.auth.login({ email });');
    console.log('  }');
  }
})();


## 1. Create a Design Project

A **Design** project is an open-source hardware design — the root of the ValueFlows graph.

In [ ]:
(async () => {
  try {
    const design = await client.resources.createProject({
      projectType: ProjectType.DESIGN,
      name: 'My Open Source Design',
      note: 'A community-driven hardware design',
      tags: ['open-source', '3d-printing', 'iot'],
      license: 'CERN-OHL-S-2.0',
      repo: 'https://github.com/interfacerproject/example-design',
      metadata: {
        version: '1.0.0',
        author: 'Community',
      },
    });
    
    console.log('✅ Design created!');
    console.log('  ID:', design.id);
    console.log('  Name:', design.name);
    
    // Save ID for later use
    window.__DESIGN_ID = design.id;
  } catch (err) {
    console.error('❌ Failed:', err.message);
    console.log('\n💡 Check that you are authenticated and the sandbox is running.');
  }
})();


## 2. Create a Product Project

**Products** are manufacturable items derived from a design.
They support product-specific filter metadata.

In [ ]:
(async () => {
  try {
    const product = await client.resources.createProject({
      projectType: ProjectType.PRODUCT,
      name: 'My 3D Printed Enclosure',
      note: 'A snap-fit enclosure for IoT boards',
      tags: ['enclosure', 'snap-fit', 'pla'],
      license: 'CERN-OHL-P-2.0',
      metadata: {
        material: 'PLA',
        printTime: '4h',
        dimensions: '100x60x30mm',
      },
    });
    
    console.log('✅ Product created!');
    console.log('  ID:', product.id);
    console.log('  Name:', product.name);
    
    window.__PRODUCT_ID = product.id;
  } catch (err) {
    console.error('❌ Failed:', err.message);
  }
})();


## 3. Create a Service Project

In [ ]:
(async () => {
  try {
    const service = await client.resources.createProject({
      projectType: ProjectType.SERVICE,
      name: 'PCB Assembly Service',
      note: 'Professional PCB assembly with 2-week turnaround',
      tags: ['pcb', 'assembly', 'electronics'],
      metadata: {
        turnaroundDays: 14,
        maxBoardsPerBatch: 50,
      },
    });
    
    console.log('✅ Service created!');
    console.log('  ID:', service.id);
    console.log('  Name:', service.name);
    
    window.__SERVICE_ID = service.id;
  } catch (err) {
    console.error('❌ Failed:', err.message);
  }
})();


## 4. Create a Machine

Machines are production tools — CNC routers, 3D printers, laser cutters.

In [ ]:
(async () => {
  try {
    const machine = await client.resources.createMachine({
      name: 'Bambu Lab X1 Carbon',
      note: 'Multi-material 3D printer',
      metadata: {
        buildVolume: '256x256x256mm',
        materials: ['PLA', 'PETG', 'ABS', 'TPU'],
        nozzleTemp: '300°C',
        bedTemp: '110°C',
      },
    });
    
    console.log('✅ Machine created!');
    console.log('  ID:', machine.id);
    console.log('  Name:', machine.name);
    
    window.__MACHINE_ID = machine.id;
  } catch (err) {
    console.error('❌ Failed:', err.message);
  }
})();


## 5. Reading Resources

In [ ]:
(async () => {
  // Get a single resource by ID
  const designId = window.__DESIGN_ID;
  if (designId) {
    const resource = await client.resources.getResource(designId);
    console.log('📋 Single resource:', resource ? 'found' : 'not found');
    if (resource) {
      console.log('  Type:', typeof resource);
      console.log('  Keys:', Object.keys(resource).slice(0, 10).join(', '));
    }
  } else {
    console.log('⚠️ No design created yet — run the previous cells first.');
  }
  // List all projects (with optional filter and pagination)
  console.log('\n📚 Fetching projects...');
  const projects = await client.resources.getProjects();
  console.log('  Projects:', projects ? 'fetched' : 'empty');
  // List all machines
  console.log('⚙️ Fetching machines...');
  const machines = await client.resources.getMachines();
  console.log('  Machines:', machines ? 'fetched' : 'empty');
  // List with pagination
  console.log('📄 Paginated list...');
  const page = await client.resources.listResources({}, { first: 10 });
  console.log('  Paginated result:', page ? 'ok' : 'empty');
})();


## 6. Resource Relations

Relate resources through Processes. Three relation types:

| Relation | Meaning |
|---|---|
| **cite** | Reference another resource (dependency) |
| **consume** | Use up a material resource |
| **contributeTo** | Add value to an existing resource |

In [ ]:
(async () => {
  const productId = window.__PRODUCT_ID;
  const designId = window.__DESIGN_ID;
  if (productId && designId) {
    // Create a process for the relation
    const processId = await client.resources.createProcess('Linking product to design');
    console.log('🔗 Process created:', processId);
    
    // Cite: the product references the design it derives from
    try {
      await client.resources.citeResource(designId, processId);
      console.log('✅ Product cites Design');
    } catch (err) {
      console.error('❌ citeResource failed:', err.message);
    }
  } else {
    console.log('⚠️ Need both product and design to create relations.');
  }
})();


## 7. Metadata & Tags

Update resource metadata, classified-as tags, and location.

In [ ]:
(async () => {
  const updateId = window.__DESIGN_ID;
  if (updateId) {
    // Update metadata
    try {
      await client.resources.updateMetadata(updateId, {
        version: '2.0.0',
        changelog: 'Added snap-fit mounting holes',
        contributors: [{ name: 'Alice', role: 'Designer' }],
      });
      console.log('✅ Metadata updated');
    } catch (err) {
      console.error('❌ updateMetadata failed:', err.message);
    }
    
    // Update tags (classifiedAs)
    try {
      await client.resources.updateClassifiedAs(updateId, [
        'tag-open-source',
        'tag-3d-printing',
        'tag-iot',
        'tag-enclosure',
      ]);
      console.log('✅ Tags updated');
    } catch (err) {
      console.error('❌ updateClassifiedAs failed:', err.message);
    }
  } else {
    console.log('⚠️ No resource to update.');
  }
})();


## 8. Location

Create a location and attach it to a resource.

In [ ]:
(async () => {
  try {
    const location = await client.resources.createLocation({
      name: 'Fab Lab Barcelona',
      address: 'Carrer de Pujades, 102, 08005 Barcelona',
      lat: 41.4015,
      lng: 2.1986,
    });
    
    console.log('📍 Location created:');
    console.log('  ID:', location.id);
    console.log('  Lat:', location.lat, 'Long:', location.long);
    
    // Relocate a resource to this location
    const relocateId = window.__DESIGN_ID;
    if (relocateId) {
      await client.resources.relocateResource(relocateId, location.id);
      console.log('✅ Resource relocated to', location.id);
    }
  } catch (err) {
    console.error('❌ Location operation failed:', err.message);
  }
})();


## 9. Proposals & Contributions

The contribution workflow: **Fork → Modify → Propose → Accept/Reject**

```mermaid
sequenceDiagram
    participant A as Author
    participant S as Zenflows
    participant O as Owner
    
    A->>S: Fork resource (create copy)
    A->>A: Modify fork
    A->>S: proposeContribution(fork, origin)
    S-->>O: Proposal received
    O->>S: acceptProposal or rejectProposal
    S-->>A: Updated resource
```

> ⚠️ Proposals require an existing resource to contribute to.

In [ ]:
(async () => {
  const originId = window.__DESIGN_ID;
  const ownerId = client.store.getItem('authId');
  if (originId && ownerId) {
    try {
      const proposal = await client.resources.proposeContribution({
        resourceForkedId: originId,  // The forked/modified copy
        resourceOriginId: originId,  // The original (demo: same resource)
        ownerId: ownerId,
        note: 'Proposed improvement: better thermal design',
      });
      
      console.log('📨 Proposal submitted!');
      console.log('  Proposal ID:', proposal.proposalId);
      console.log('  Cite Intent:', proposal.citeIntentId);
      console.log('  Accept Intent:', proposal.acceptIntentId);
      console.log('  Modify Intent:', proposal.modifyIntentId);
      
      // Store for accept/reject demo
      window.__PROPOSAL = proposal;
    } catch (err) {
      console.error('❌ proposeContribution failed:', err.message);
      console.log('\n💡 This is expected on sandbox — proposals need specific resource state.');
    }
  } else {
    console.log('⚠️ No resources to propose against.');
  }
})();


### Accept or Reject a Proposal

Run the appropriate cell below:

In [ ]:
(async () => {
  // Accept the proposal
  const proposal = window.__PROPOSAL;
  const originId = window.__DESIGN_ID;
  const ownerId = client.store.getItem('authId');
  if (proposal && originId && ownerId) {
    try {
      await client.resources.acceptProposal({
        proposalId: proposal.proposalId,
        citeIntentId: proposal.citeIntentId,
        acceptIntentId: proposal.acceptIntentId,
        modifyIntentId: proposal.modifyIntentId,
        resourceForkedId: originId,
        resourceOriginId: originId,
        ownerId: ownerId,
        proposerId: ownerId,
        newMetadata: { acceptedProposal: true, mergedAt: new Date().toISOString() },
      });
      console.log('✅ Proposal accepted!');
    } catch (err) {
      console.error('❌ acceptProposal failed:', err.message);
    }
  }
})();


In [ ]:
(async () => {
  // Reject the proposal
  const proposal = window.__PROPOSAL;
  if (proposal) {
    try {
      await client.resources.rejectProposal(
        proposal.citeIntentId,
        proposal.acceptIntentId,
        proposal.modifyIntentId
      );
      console.log('🚫 Proposal rejected.');
    } catch (err) {
      console.error('❌ rejectProposal failed:', err.message);
    }
  }
})();


## 📋 API Reference: ResourceClient

| Method | Description |
|---|---|
| `createProcess(name)` | Create a workflow process |
| `createLocation({ name, address, lat, lng })` | Create a spatial location |
| `createProject({ projectType, name, note, tags, license, ... })` | Create Design/Product/Service |
| `createMachine({ name, note, metadata })` | Create a machine resource |
| `createDppResource({ name, dppUlid })` | Create a DPP-linked resource |
| `getResource(id)` | Fetch a single resource |
| `getProjects(filter?, pagination?)` | List projects |
| `getMachines()` | List machines |
| `listResources(filter?, pagination?)` | Paginated resource list |
| `citeResource(resourceId, processId)` | Reference another resource |
| `consumeResource(resourceId, processId)` | Consume a material resource |
| `contributeToResource(processId, specId)` | Add value to a resource |
| `updateMetadata(resourceId, metadata)` | Update JSON metadata |
| `updateClassifiedAs(resourceId, tags)` | Update classification tags |
| `relocateResource(resourceId, locationId)` | Move to new location |
| `proposeContribution({ ... })` | Create a contribution proposal |
| `acceptProposal({ ... })` | Accept a proposal |
| `rejectProposal(intent1, intent2, intent3)` | Reject a proposal |

## ✅ Summary

You've learned:
- ✅ ValueFlows concepts and the Facade pattern
- ✅ Creating Design, Product, Service, and Machine resources
- ✅ Reading and listing resources with pagination
- ✅ Resource relations (cite, consume, contribute)
- ✅ Metadata updates and tag management
- ✅ Location creation and relocation
- ✅ Contribution proposals (fork → propose → accept/reject)

**Next:** `03_Files_and_Hashing.ipynb` — cryptographic file handling!